# Training a Diabetes Prediction Model - Logistic Regression

## Setup the Azure ML Client

In [10]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

Found the config file in: /config.json
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


## Loading the Diabetes Dataset from the Data Asset

In [11]:
import mltable

tbl = mltable.load("azureml:diabetes-mltable:1")
diabetes_df = tbl.to_pandas_dataframe()

diabetes_df.head()

Found the config file in: /config.json
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Overriding of current MeterProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,True
1,1,85,66,29,0,26.6,0.351,31,False
2,8,183,64,0,0,23.3,0.672,32,True
3,1,89,66,23,94,28.1,0.167,21,False
4,0,137,40,35,168,43.1,2.288,33,True


## Normalizing and Scaling the Data

In [12]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

numericFeatures = [
    "Pregnancies", "Glucose", "BloodPressure",
    "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction"
]

scaler = MinMaxScaler()
normalizedFeatures = scaler.fit_transform(diabetes_df[numericFeatures])


In [14]:
numericFeaturesVector = diabetes_df[numericFeatures]

# Compare original vs scaled
compareNumerics = pd.DataFrame({
    "numericFeatures": numericFeaturesVector.values.tolist(),
    "normalizedFeatures": normalizedFeatures.tolist()
})

display(compareNumerics)

,numericFeatures,normalizedFeatures
0,"[6.0, 148.0, 72.0, 35.0, 0.0, 33.6, 0.627]","[0.3529411764705882, 0.7437185929648241, 0.590..."
1,"[1.0, 85.0, 66.0, 29.0, 0.0, 26.6, 0.351]","[0.058823529411764705, 0.4271356783919598, 0.5..."
2,"[8.0, 183.0, 64.0, 0.0, 0.0, 23.3, 0.672]","[0.47058823529411764, 0.9195979899497487, 0.52..."
3,"[1.0, 89.0, 66.0, 23.0, 94.0, 28.1, 0.167]","[0.058823529411764705, 0.4472361809045226, 0.5..."
4,"[0.0, 137.0, 40.0, 35.0, 168.0, 43.1, 2.288]","[0.0, 0.6884422110552764, 0.3278688524590164, ..."
...,...,...
763,"[10.0, 101.0, 76.0, 48.0, 180.0, 32.9, 0.171]","[0.5882352941176471, 0.507537688442211, 0.6229..."
764,"[2.0, 122.0, 70.0, 27.0, 0.0, 36.8, 0.34]","[0.11764705882352941, 0.6130653266331658, 0.57..."
765,"[5.0, 121.0, 72.0, 23.0, 112.0, 26.2, 0.245]","[0.29411764705882354, 0.6080402010050251, 0.59..."
766,"[1.0, 126.0, 60.0, 0.0, 0.0, 30.1, 0.349]","[0.058823529411764705, 0.6331658291457286, 0.4..."


In [16]:
from sklearn.model_selection import train_test_split

scaledData = pd.DataFrame(normalizedFeatures, columns=numericFeatures)
scaledData["Outcome"] = diabetes_df["Outcome"]

train, test = train_test_split(scaledData, test_size=0.3, random_state=42)
print("Training Rows:", len(train), "Testing Rows:", len(test))

X_train = train[numericFeatures]
Y_train = train["Outcome"]
X_test = test[numericFeatures]
Y_test = test["Outcome"]

Training Rows: 537 Testing Rows: 231


In [17]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=10, C=1/0.3)
model.fit(X_train, Y_train)

print("Model trained!")

Model trained!


In [18]:
from sklearn.metrics import classification_report

predictions = model.predict(X_test)
print(classification_report(Y_test, predictions))

              precision    recall  f1-score   support

       False       0.80      0.85      0.82       151
        True       0.67      0.59      0.63        80

    accuracy                           0.76       231
   macro avg       0.73      0.72      0.72       231
weighted avg       0.75      0.76      0.75       231

